# ASTE Pipeline Model - Version 6

This is a high-precision, multi-stage pipeline for Aspect Sentiment Triplet Extraction (ASTE) using `microsoft/deberta-v3-base`.

## Architecture Overview

1. **Stage 1**: Joint Span & BIO Extractor. DeBERTa encoder with BIO token classification head and span-pair classification head.
2. **Stage 2**: Pair Classification. Encoder receives `[CLS] + Aspect + Opinion`. Explicit features included: `distance`, `order`, and `has_but`.
3. **Stage 3**: Dual-Head VA Regression/Classification. Regression without dropout. Classification with dropout (0.2). Predicts Valence and Arousal scores, optimizing for explicit VA-T-F1 integer mapping.


In [1]:
import json
import warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from typing import List, Dict, Tuple, Optional
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModel,
    get_linear_schedule_with_warmup,
    logging as hf_logging,
)
from torch.optim import AdamW
import re

warnings.filterwarnings('ignore')
hf_logging.set_verbosity_error()

# Paths
DATA_DIR    = Path('/kaggle/input/datasets/chaitanyajx1/datasetnlp')
REST_FILE   = DATA_DIR / 'restaurant_train.jsonl'
LAP_FILE    = DATA_DIR / 'laptop_train.jsonl'
TEST_FILE   = DATA_DIR / 'test.jsonl'

# Fallback paths for local testing
if not DATA_DIR.exists():
    DATA_DIR    = Path('.')
    REST_FILE   = DATA_DIR / 'restaurant_train.jsonl'
    LAP_FILE    = DATA_DIR / 'laptop_train.jsonl'
    TEST_FILE   = DATA_DIR / 'test.jsonl'

MODEL_NAME  = 'microsoft/deberta-v3-base'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}, Model: {MODEL_NAME}")


Device: cuda, Model: microsoft/deberta-v3-base


## 1. Data Pre-Processing Data & Feature Engineering
Extracting BIO tags, normalizing pairs, computing explicit features (`distance`, `order`, `has_but`).

In [2]:
def get_features(text: str, aspect: str, opinion: str) -> Tuple[float, float, float]:
    """
    Explicit feature computation:
    - distance (normalized between 0 and 1, assuming max sentence len 128)
    - order (1 if aspect is before opinion, else 0)
    - has_but (1 if 'but', 'however', 'although' occurs between them, else 0)
    """
    a_i = text.lower().find(aspect.lower())
    o_i = text.lower().find(opinion.lower())
    if a_i == -1 or o_i == -1:
        return 0.0, 0.0, 0.0
    
    dist = abs(a_i - o_i) / 128.0
    order = 1.0 if a_i < o_i else 0.0
    
    # Check for contrastive words between aspect and opinion
    start_idx = min(a_i, o_i)
    end_idx = max(a_i, o_i)
    middle_text = text[start_idx:end_idx].lower()
    has_but = 1.0 if any(b in middle_text for b in ['but', 'however', 'although']) else 0.0
    
    return dist, order, has_but

# Validation Lexicon Prior Loader
def load_nrc_prior(path='nrc_vad.txt'):
    lexicon = {}
    if Path(path).exists():
        with open(path, 'r', encoding='utf-8') as f:
            for line in f:
                parts = line.strip().split('\t')
                if len(parts) == 4:
                    lexicon[parts[0]] = (float(parts[1]), float(parts[2]), float(parts[3]))
    return lexicon


## 2. Stage 1: Joint BIO & Span Extractor Model
Uses a single `JointStage1Model` for sharing DeBERTa signals to both BIO tagging and pair tagging.

In [3]:
class JointStage1Model(nn.Module):
    def __init__(self, model_name=MODEL_NAME, num_tags=5):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size
        
        self.dropout = nn.Dropout(0.1)
        # BIO Head
        self.bio_head = nn.Linear(hidden_size, num_tags)
        # Span Pair Classifier Head
        self.span_head = nn.Sequential(
            nn.Linear(hidden_size * 2 + 32, 256),  # concat(h_i, h_j, width_emb)
            nn.ReLU(),
            nn.LayerNorm(256),
            nn.Dropout(0.1),
            nn.Linear(256, 1)
        )
        self.width_embedding = nn.Embedding(128, 32)
        
    def forward(self, input_ids, attention_mask, span_pairs=None):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        hidden = self.dropout(outputs.last_hidden_state)
        
        bio_logits = self.bio_head(hidden)
        
        span_logits = None
        if span_pairs is not None:
            # Minimal dummy logic for span processing. Expand for real loop.
            B = hidden.size(0)
            span_logits = []
            for b in range(B):
                s_logits_b = []
                for (i, j) in span_pairs[b]:
                    h_i, h_j = hidden[b, i], hidden[b, j]
                    w_emb = self.width_embedding(torch.tensor(abs(i-j), device=hidden.device))
                    s_logits_b.append(self.span_head(torch.cat([h_i, h_j, w_emb])))
                if len(s_logits_b) > 0:
                    span_logits.append(torch.stack(s_logits_b))
                else:
                    span_logits.append(torch.empty(0, 1, device=hidden.device))
            span_logits = span_logits
            
        return {"bio_logits": bio_logits, "span_logits": span_logits, "hidden": hidden}


## 3. Stage 2 & 3: Pair Classification and Dual-Head VA Engine
Connects pairing via CLS with specific context inputs, and extracts exact discrete classification and zero-dropout regression.

In [4]:
class Stage2_3_Model(nn.Module):
    def __init__(self, model_name=MODEL_NAME):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size
        
        # Stage 2: Pair Classification Head
        # Features concatenated: h[CLS] + 3 explicit signals (distance, order, has_but)
        self.pair_classifier = nn.Sequential(
            nn.Linear(hidden_size + 3, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )
        
        # Stage 3: Dual-Head VA 
        # Regression head (no dropout for precision)
        self.va_regress_v = nn.Sequential(
            nn.Linear(hidden_size, 128), nn.GELU(), nn.Linear(128, 1)
        )
        self.va_regress_a = nn.Sequential(
            nn.Linear(hidden_size, 128), nn.GELU(), nn.Linear(128, 1)
        )
        
        # Classification head (with dropout)
        self.va_class_v = nn.Sequential(
            nn.Dropout(0.2), nn.Linear(hidden_size, 9) # 9 discrete buckets (1-9)
        )
        self.va_class_a = nn.Sequential(
            nn.Dropout(0.2), nn.Linear(hidden_size, 9)
        )
        
    def forward(self, input_ids, attention_mask, explicit_features):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        # Extract [CLS] representing: [CLS] Aspect [SEP] Opinion [SEP] Context
        cls_repr = outputs.last_hidden_state[:, 0, :]
        
        # Concat explicit features for Stage 2
        pair_input = torch.cat([cls_repr, explicit_features], dim=-1)
        pair_logits = self.pair_classifier(pair_input)
        
        # Va Heads
        reg_v = self.va_regress_v(cls_repr)
        reg_a = self.va_regress_a(cls_repr)
        
        class_v = self.va_class_v(cls_repr)
        class_a = self.va_class_a(cls_repr)
        
        return {
            "pair_logits": pair_logits,
            "reg_v": reg_v, "reg_a": reg_a,
            "class_v": class_v, "class_a": class_a
        }

    def get_final_va(self, reg_v_pred, class_v_probs):
        """
        (round(Reg_pred) + Class_pred) / 2 logic for exact integer metric.
        """
        class_v_pred = torch.argmax(class_v_probs, dim=-1) + 1 # offset from 0-mapped indices
        final_v = (torch.round(reg_v_pred) + class_v_pred) / 2.0
        return final_v
print('Model architecture initialized!')


Model architecture initialized!


In [1]:
"""
ASTE (Aspect Sentiment Triplet Extraction) with Valence-Arousal Regression
==========================================================================
Architecture:
  - DeBERTa-v3-base encoder (shared)
  - Stage 1: Joint BIO tagger + Aspect span head + Opinion span head
  - Stage 2: Sentence-aware pairing classifier (with has_but feature)
  - Stage 3: Dual VA heads (regression + classification, averaged)
  - Evaluation: VA-T-F1 (round(V) + round(A) + exact span match)

Usage on Kaggle:
  !pip install transformers==4.40.0 torch --quiet
  Then run: python aste_model.py
"""

import json
import math
import random
import re
from collections import defaultdict
from itertools import product
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import (
    AutoModel,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
)

# ── Paths (Kaggle) ──────────────────────────────────────────────────────────
DATA_DIR  = Path("/kaggle/input/datasets/chaitanyajx1/datasetnlp")
REST_FILE = DATA_DIR / "restaurant_train.jsonl"
LAP_FILE  = DATA_DIR / "laptop_train.jsonl"

# ── Config ───────────────────────────────────────────────────────────────────
CFG = {
    "model_name":        "microsoft/deberta-v3-base",   # swap to -large if GPU ≥ 24GB
    "max_len":           128,
    "hidden_size":       768,        # 1024 for deberta-v3-large
    "width_emb_size":    64,
    "span_max_len":      8,
    "bio_classes":       5,          # O B-Asp I-Asp B-Opi I-Opi
    "num_bins":          32,         # VA classification bins (0.25 width each)
    "bio_lr":            2e-5,
    "head_lr":           5e-4,
    "weight_decay":      0.01,
    "warmup_ratio":      0.1,
    "batch_size":        16,
    "epochs_extract":    20,
    "epochs_pair":       15,
    "epochs_va":         15,
    "pair_threshold":    0.5,
    "span_threshold":    0.5,
    "bio_proposal_thr":  0.3,        # soft BIO proposal threshold
    "seed":              42,
    "device":            "cuda" if torch.cuda.is_available() else "cpu",
    "neg_ratio":         3,          # negative pairs per positive pair
}

CONTRASTIVE_WORDS = {"but", "however", "although", "though", "yet", "while",
                     "whereas", "except", "despite", "still", "yet"}

BIO_O, BIO_B_ASP, BIO_I_ASP, BIO_B_OPI, BIO_I_OPI = 0, 1, 2, 3, 4
BIO_LABEL_MAP = {"O": 0, "B-Asp": 1, "I-Asp": 2, "B-Opi": 3, "I-Opi": 4}


# ── Reproducibility ──────────────────────────────────────────────────────────
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

In [15]:
# ═══════════════════════════════════════════════════════════════════════════
# 1.  DATA LOADING & PRE-PROCESSING
# ═══════════════════════════════════════════════════════════════════════════

def load_jsonl(path: Path):
    records = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


def find_span_tokens(text: str, span: str, encoding):
    """Return (tok_start, tok_end) inclusive token indices for span in text."""
    if span == "NULL":
        return None
    char_start = text.find(span)
    if char_start == -1:
        # try case-insensitive
        lower_pos = text.lower().find(span.lower())
        if lower_pos == -1:
            return None
        char_start = lower_pos
    char_end = char_start + len(span) - 1  # inclusive

    offsets = encoding["offset_mapping"]
    tok_start, tok_end = None, None
    for i, (cs, ce) in enumerate(offsets):
        if cs == 0 and ce == 0:
            continue
        if tok_start is None and cs <= char_start < ce:
            tok_start = i
        if cs <= char_end < ce:
            tok_end = i
    if tok_start is None or tok_end is None:
        return None
    return (tok_start, tok_end)


def build_bio_labels(seq_len: int, asp_spans, opi_spans):
    labels = [BIO_O] * seq_len
    for (s, e) in (asp_spans or []):
        if s is None:
            continue
        labels[s] = BIO_B_ASP
        for i in range(s + 1, e + 1):
            labels[i] = BIO_I_ASP
    for (s, e) in (opi_spans or []):
        if s is None:
            continue
        labels[s] = BIO_B_OPI
        for i in range(s + 1, e + 1):
            labels[i] = BIO_I_OPI
    return labels


def va_to_bin(score: float) -> int:
    """Convert VA score [1,9] to bin index [0,31]."""
    return max(0, min(31, int((score - 1.0) / 0.25)))


def bin_to_score(bin_idx: int) -> float:
    """Convert bin index back to midpoint score."""
    return 1.0 + bin_idx * 0.25 + 0.125


def parse_va(va_str: str):
    """Parse 'V#A' string to (float, float)."""
    parts = va_str.split("#")
    return float(parts[0]), float(parts[1])


def has_but_between(tokens, start: int, end: int) -> float:
    """1.0 if a contrastive word exists between token positions start and end."""
    lo, hi = min(start, end), max(start, end)
    for i in range(lo + 1, hi):
        if i < len(tokens) and tokens[i].lower() in CONTRASTIVE_WORDS:
            return 1.0
    return 0.0


def process_records(records, domain: str, tokeniser, max_len: int):
    """
    Returns list of processed dicts, one per sentence:
      - input_ids, attention_mask: [max_len]
      - bio_labels:                [max_len]
      - gold_asp_spans:            list of (tok_start, tok_end)
      - gold_opi_spans:            list of (tok_start, tok_end)
      - gold_pairs:                list of (asp_tok_span, opi_tok_span, V, A)
      - raw_text, id
    """
    prefix = "[NULL] [LAP] " if domain == "lap" else "[NULL] [REST] "
    processed = []

    for rec in records:
        text   = rec["Text"].strip()
        aug    = prefix + text
        quads  = rec.get("Quadruplet", [])

        enc = tokeniser(
            aug,
            max_length=max_len,
            padding="max_length",
            truncation=True,
            return_offsets_mapping=True,
            return_tensors="pt",
        )

        input_ids      = enc["input_ids"].squeeze(0)
        attention_mask = enc["attention_mask"].squeeze(0)
        offsets        = enc["offset_mapping"].squeeze(0).tolist()
        tokens_decoded = [tokeniser.decode([i]) for i in input_ids.tolist()]

        asp_spans, opi_spans, gold_pairs = [], [], []

        for quad in quads:
            asp_text = quad.get("Aspect", "NULL")
            opi_text = quad.get("Opinion", "NULL")
            va_str   = quad.get("VA", "5.0#5.0")
            V, A     = parse_va(va_str)

            if asp_text == "NULL":
                # tag the [NULL] token (position 0 after prepend = position 2 in tokenised)
                asp_span = (0, 0)  # [NULL] token position
            else:
                asp_span = find_span_tokens(aug, asp_text,
                                            {"offset_mapping": offsets})

            if opi_text == "NULL":
                opi_span = (0, 0)
            else:
                opi_span = find_span_tokens(aug, opi_text,
                                            {"offset_mapping": offsets})

            if asp_span is not None:
                asp_spans.append(asp_span)
            if opi_span is not None:
                opi_spans.append(opi_span)
            if asp_span is not None and opi_span is not None:
                gold_pairs.append((asp_span, opi_span, V, A))

        bio_labels = build_bio_labels(max_len, asp_spans, opi_spans)

        processed.append({
            "id":              rec.get("ID", ""),
            "raw_text":        text,
            "aug_text":        aug,
            "input_ids":       input_ids,
            "attention_mask":  attention_mask,
            "bio_labels":      torch.tensor(bio_labels, dtype=torch.long),
            "gold_asp_spans":  asp_spans,
            "gold_opi_spans":  opi_spans,
            "gold_pairs":      gold_pairs,
            "offsets":         offsets,
            "tokens_decoded":  tokens_decoded,
            "domain":          domain,
        })

    return processed


# ── Negative pair construction ───────────────────────────────────────────────

def build_negative_pairs(all_records, neg_ratio: int = 3):
    """
    Build negative (asp_text, opi_text, sentence, domain) examples.
    Strategy A: cross-sentence (different sentence indices)
    Strategy B: within-sentence wrong combos (non-gold Cartesian product)
    Strategy C: boundary perturbation (extend/shrink span by 1 token)
    """
    positives, negatives = [], []

    # collect all positives first
    for rec in all_records:
        for (asp_span, opi_span, V, A) in rec["gold_pairs"]:
            positives.append({
                "aug_text":       rec["aug_text"],
                "input_ids":      rec["input_ids"],
                "attention_mask": rec["attention_mask"],
                "asp_span":       asp_span,
                "opi_span":       opi_span,
                "tokens_decoded": rec["tokens_decoded"],
                "label":          1,
                "V":              V,
                "A":              A,
            })

    # Strategy B: within-sentence wrong combos
    for rec in all_records:
        asps = rec["gold_asp_spans"]
        opis = rec["gold_opi_spans"]
        gp_set = {(a, o) for (a, o, _, _) in rec["gold_pairs"]}
        for asp in asps:
            for opi in opis:
                if (asp, opi) not in gp_set:
                    negatives.append({
                        "aug_text":       rec["aug_text"],
                        "input_ids":      rec["input_ids"],
                        "attention_mask": rec["attention_mask"],
                        "asp_span":       asp,
                        "opi_span":       opi,
                        "tokens_decoded": rec["tokens_decoded"],
                        "label":          0,
                        "V":              5.0,
                        "A":              5.0,
                    })

    # Strategy A: cross-sentence negatives
    n_cross = max(0, len(positives) * neg_ratio - len(negatives))
    for _ in range(n_cross):
        r1 = random.choice(all_records)
        r2 = random.choice(all_records)
        if r1["gold_asp_spans"] and r2["gold_opi_spans"]:
            asp = random.choice(r1["gold_asp_spans"])
            opi = random.choice(r2["gold_opi_spans"])
            negatives.append({
                "aug_text":       r1["aug_text"],
                "input_ids":      r1["input_ids"],
                "attention_mask": r1["attention_mask"],
                "asp_span":       asp,
                "opi_span":       opi,
                "tokens_decoded": r1["tokens_decoded"],
                "label":          0,
                "V":              5.0,
                "A":              5.0,
            })

    return positives + negatives

In [16]:
# ═══════════════════════════════════════════════════════════════════════════
# 2.  DATASETS
# ═══════════════════════════════════════════════════════════════════════════

class ExtractionDataset(Dataset):
    """Dataset for Stage 1: BIO + span scoring."""
    def __init__(self, records):
        self.records = records

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]
        return {
            "input_ids":      rec["input_ids"],
            "attention_mask": rec["attention_mask"],
            "bio_labels":     rec["bio_labels"],
            "gold_asp_spans": rec["gold_asp_spans"],
            "gold_opi_spans": rec["gold_opi_spans"],
            "idx":            idx,
        }


class PairingDataset(Dataset):
    """Dataset for Stage 2: pairing classifier."""
    def __init__(self, pair_examples, tokeniser, max_len):
        self.examples = pair_examples
        self.tokeniser = tokeniser
        self.max_len = max_len

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        ex = self.examples[idx]
        asp_s, asp_e = ex["asp_span"]
        opi_s, opi_e = ex["opi_span"]
        toks = ex["tokens_decoded"]

        asp_text = "".join(toks[asp_s: asp_e + 1]).strip()
        opi_text = "".join(toks[opi_s: opi_e + 1]).strip()

        # Build "[sentence] [SEP] asp_text [SEP] opi_text"
        combined = ex["aug_text"] + " [SEP] " + asp_text + " [SEP] " + opi_text
        enc = self.tokeniser(
            combined,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )

        # hand-crafted features
        dist = abs(asp_s - opi_s) / max(self.max_len, 1)
        order = 1.0 if asp_s < opi_s else 0.0
        hb = has_but_between(toks, asp_s, opi_s)

        return {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "dist":           torch.tensor([dist],  dtype=torch.float),
            "order":          torch.tensor([order], dtype=torch.float),
            "has_but":        torch.tensor([hb],    dtype=torch.float),
            "label":          torch.tensor(ex["label"], dtype=torch.long),
            "V":              torch.tensor(ex["V"], dtype=torch.float),
            "A":              torch.tensor(ex["A"], dtype=torch.float),
        }


class VADataset(Dataset):
    """Dataset for Stage 3: VA heads (only positive pairs)."""
    def __init__(self, pair_examples, tokeniser, max_len):
        self.examples = [e for e in pair_examples if e["label"] == 1]
        self.tokeniser = tokeniser
        self.max_len = max_len

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        ex = self.examples[idx]
        asp_s, asp_e = ex["asp_span"]
        opi_s, opi_e = ex["opi_span"]
        toks = ex["tokens_decoded"]

        asp_text = "".join(toks[asp_s: asp_e + 1]).strip()
        opi_text = "".join(toks[opi_s: opi_e + 1]).strip()

        combined = ex["aug_text"] + " [SEP] " + asp_text + " [SEP] " + opi_text
        enc = self.tokeniser(
            combined,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )

        V, A = ex["V"], ex["A"]
        return {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "V":              torch.tensor(V, dtype=torch.float),
            "A":              torch.tensor(A, dtype=torch.float),
            "V_bin":          torch.tensor(va_to_bin(V), dtype=torch.long),
            "A_bin":          torch.tensor(va_to_bin(A), dtype=torch.long),
        }

In [17]:
# ═══════════════════════════════════════════════════════════════════════════
# 3.  MODEL COMPONENTS
# ═══════════════════════════════════════════════════════════════════════════

class ExtractionModel(nn.Module):
    """
    Stage 1: shared DeBERTa + BIO head + Aspect span head + Opinion span head.
    One forward pass, three losses summed.
    """
    def __init__(self, cfg):
        super().__init__()
        self.encoder     = AutoModel.from_pretrained(cfg["model_name"])
        H                = cfg["hidden_size"]
        W                = cfg["width_emb_size"]
        self.width_emb   = nn.Embedding(cfg["span_max_len"] + 1, W)
        self.bio_head    = nn.Linear(H, cfg["bio_classes"])
        self.asp_head    = nn.Sequential(nn.Linear(2 * H + W, 256), nn.ReLU(),
                                         nn.Dropout(0.3), nn.Linear(256, 1))
        self.opi_head    = nn.Sequential(nn.Linear(2 * H + W, 256), nn.ReLU(),
                                         nn.Dropout(0.3), nn.Linear(256, 1))

    def encode(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        return out.last_hidden_state  # [B, L, H]

    def span_vec(self, h, i, j):
        """Build span vector for token range [i,j]."""
        # Max index should be CFG["span_max_len"] - 1. Also prevent negatives.
        length = max(0, min(j - i, CFG["span_max_len"] - 1)) 
        w = self.width_emb(torch.tensor(length, device=h.device))
        return torch.cat([h[i], h[j], w], dim=-1)

    def forward(self, input_ids, attention_mask, bio_labels=None,
                gold_asp_spans=None, gold_opi_spans=None):
        h = self.encode(input_ids, attention_mask)  # [B, L, H]
        h = h.float()  # forced float32
        
        # BIO loss
        bio_logits = self.bio_head(h)               # [B, L, 5]
        loss_bio = torch.tensor(0.0, device=h.device)
        if bio_labels is not None:
            loss_bio = F.cross_entropy(
                bio_logits.view(-1, CFG["bio_classes"]),
                bio_labels.view(-1),
                ignore_index=-100,
            )

        # Span losses (computed per example in batch)
        loss_asp = torch.tensor(0.0, device=h.device)
        loss_opi = torch.tensor(0.0, device=h.device)
        asp_count = opi_count = 0

        if gold_asp_spans is not None and gold_opi_spans is not None:
            B = h.size(0)
            for b in range(B):
                h_b = h[b]  # [L, H]

                # Aspect spans
                for (si, ei) in gold_asp_spans[b]:
                    sv = self.span_vec(h_b, si, ei)
                    score = torch.sigmoid(self.asp_head(sv.unsqueeze(0)).squeeze())
                    loss_asp = loss_asp + F.binary_cross_entropy(
                        score, torch.ones(1, device=h.device).squeeze())
                    asp_count += 1

                # Opinion spans
                for (si, ei) in gold_opi_spans[b]:
                    sv = self.span_vec(h_b, si, ei)
                    score = torch.sigmoid(self.opi_head(sv.unsqueeze(0)).squeeze())
                    loss_opi = loss_opi + F.binary_cross_entropy(
                        score, torch.ones(1, device=h.device).squeeze())
                    opi_count += 1

            if asp_count > 0:
                loss_asp = loss_asp / asp_count
            if opi_count > 0:
                loss_opi = loss_opi / opi_count

        total_loss = loss_bio + loss_asp + loss_opi
        return total_loss, bio_logits

    def predict_spans(self, input_ids, attention_mask, threshold=0.5, bio_thr=0.3):
        """Run inference: return (asp_spans, opi_spans) as token index pairs."""
        with torch.no_grad():
            h = self.encode(input_ids, attention_mask)  # [1, L, H]
            h = h.float()  # forced float32
            h = h.squeeze(0)  # [L, H]
            bio_logits = self.bio_head(h)  # [L, 5]
            bio_probs  = torch.softmax(bio_logits, dim=-1)  # [L, 5]

        L = h.size(0)
        candidates_asp, candidates_opi = [], []

        # Soft BIO proposals (recall stage)
        for i in range(L):
            if bio_probs[i, BIO_B_ASP].item() > bio_thr:
                for j in range(i, min(i + CFG["span_max_len"], L)):
                    candidates_asp.append((i, j))
            if bio_probs[i, BIO_B_OPI].item() > bio_thr:
                for j in range(i, min(i + CFG["span_max_len"], L)):
                    candidates_opi.append((i, j))

        # Span scorer (precision stage)
        final_asp, final_opi = [], []
        with torch.no_grad():
            for (si, ei) in set(candidates_asp):
                sv = self.span_vec(h, si, ei)
                sc = torch.sigmoid(self.asp_head(sv.unsqueeze(0)).squeeze()).item()
                if sc >= threshold:
                    final_asp.append((si, ei))
            for (si, ei) in set(candidates_opi):
                sv = self.span_vec(h, si, ei)
                sc = torch.sigmoid(self.opi_head(sv.unsqueeze(0)).squeeze()).item()
                if sc >= threshold:
                    final_opi.append((si, ei))

        return final_asp, final_opi


class PairingModel(nn.Module):
    """
    Stage 2: sentence-aware pairing classifier.
    Input: sentence + aspect + opinion as one string → h[CLS] + [dist, order, has_but].
    """
    def __init__(self, cfg):
        super().__init__()
        self.encoder  = AutoModel.from_pretrained(cfg["model_name"])
        H             = cfg["hidden_size"]
        self.classifier = nn.Linear(H + 3, 2)

    def forward(self, input_ids, attention_mask, dist, order, has_but, labels=None):
        out  = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls  = out.last_hidden_state[:, 0, :]          # [B, H]
        cls  = cls.float()  # forced float32
        feats = torch.cat([cls, dist, order, has_but], dim=-1)  # [B, H+3]
        logits = self.classifier(feats)                # [B, 2]

        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits, labels)

        return loss, logits


class VARegressionModel(nn.Module):
    """
    Stage 3A: VA regression with dropout DISABLED.
    Predicts raw V and A scalars in [1,9] range.
    """
    def __init__(self, cfg):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(cfg["model_name"])
        H = cfg["hidden_size"]
        self.v_head = nn.Linear(H, 1)
        self.a_head = nn.Linear(H, 1)
        self._disable_dropout()

    def _disable_dropout(self):
        for m in self.encoder.modules():
            if isinstance(m, nn.Dropout):
                m.p = 0.0

    def forward(self, input_ids, attention_mask, V=None, A=None):
        out  = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls  = out.last_hidden_state[:, 0, :]   # [B, H]
        cls  = cls.float()  # forced float32
        v_hat = self.v_head(cls).squeeze(-1)    # [B]
        a_hat = self.a_head(cls).squeeze(-1)    # [B]

        loss = None
        if V is not None and A is not None:
            loss = F.mse_loss(v_hat, V) + F.mse_loss(a_hat, A)

        return loss, v_hat, a_hat


class VAClassificationModel(nn.Module):
    """
    Stage 3B: VA classification into 32 bins of width 0.25.
    Dropout kept enabled (classification is robust to noise).
    """
    def __init__(self, cfg):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(cfg["model_name"])
        H = cfg["hidden_size"]
        self.v_head = nn.Linear(H, cfg["num_bins"])
        self.a_head = nn.Linear(H, cfg["num_bins"])

    def forward(self, input_ids, attention_mask, V_bin=None, A_bin=None):
        out  = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls  = out.last_hidden_state[:, 0, :]   # [B, H]
        cls  = cls.float()  # forced float32
        v_logits = self.v_head(cls)             # [B, 32]
        a_logits = self.a_head(cls)             # [B, 32]

        loss = None
        if V_bin is not None and A_bin is not None:
            loss = F.cross_entropy(v_logits, V_bin) + F.cross_entropy(a_logits, A_bin)

        return loss, v_logits, a_logits


In [18]:
# ═══════════════════════════════════════════════════════════════════════════
# 4.  TRAINING HELPERS
# ═══════════════════════════════════════════════════════════════════════════

def make_optimizer(model, enc_lr, head_lr, weight_decay):
    """Two param groups: encoder (small LR) and heads (large LR)."""
    enc_params  = list(model.encoder.parameters())
    enc_ids     = {id(p) for p in enc_params}
    head_params = [p for p in model.parameters() if id(p) not in enc_ids]
    return torch.optim.AdamW(
        [{"params": enc_params,  "lr": enc_lr},
         {"params": head_params, "lr": head_lr}],
        weight_decay=weight_decay,
    )


def train_extraction(model, records, cfg, out_path="model_extract.pt"):
    """Train Stage 1: BIO + span heads jointly."""
    device = cfg["device"]
    model.to(device)

    # simple collate: skip span label batching (handled per-example in forward)
    def collate(batch):
        return {
            "input_ids":      torch.stack([b["input_ids"] for b in batch]),
            "attention_mask": torch.stack([b["attention_mask"] for b in batch]),
            "bio_labels":     torch.stack([b["bio_labels"] for b in batch]),
            "gold_asp_spans": [b["gold_asp_spans"] for b in batch],
            "gold_opi_spans": [b["gold_opi_spans"] for b in batch],
        }

    dataset = ExtractionDataset(records)
    loader  = DataLoader(dataset, batch_size=cfg["batch_size"],
                         shuffle=True, collate_fn=collate)
    optim   = make_optimizer(model, cfg["bio_lr"], cfg["head_lr"], cfg["weight_decay"])
    total_steps   = cfg["epochs_extract"] * len(loader)
    warmup_steps  = int(cfg["warmup_ratio"] * total_steps)
    scheduler     = get_linear_schedule_with_warmup(optim, warmup_steps, total_steps)

    best_loss = float("inf")
    for epoch in range(cfg["epochs_extract"]):
        model.train()
        total_loss = 0.0
        for batch in loader:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            bio_labels     = batch["bio_labels"].to(device)
            gold_asp       = batch["gold_asp_spans"]
            gold_opi       = batch["gold_opi_spans"]

            loss, _ = model(input_ids, attention_mask, bio_labels, gold_asp, gold_opi)
            optim.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optim.step()
            scheduler.step()
            total_loss += loss.item()

        avg = total_loss / len(loader)
        print(f"  [Extraction] epoch {epoch+1}/{cfg['epochs_extract']}  loss={avg:.4f}")
        if avg < best_loss:
            best_loss = avg
            torch.save(model.state_dict(), out_path)

    print(f"  Best extraction loss: {best_loss:.4f}  saved to {out_path}")
    return model


def train_pairing(model, pair_examples, tokeniser, cfg, out_path="model_pair.pt"):
    """Train Stage 2: pairing classifier."""
    device  = cfg["device"]
    model.to(device)
    dataset = PairingDataset(pair_examples, tokeniser, cfg["max_len"])
    loader  = DataLoader(dataset, batch_size=cfg["batch_size"], shuffle=True)
    optim   = make_optimizer(model, cfg["bio_lr"], cfg["head_lr"], cfg["weight_decay"])
    total_steps  = cfg["epochs_pair"] * len(loader)
    warmup_steps = int(cfg["warmup_ratio"] * total_steps)
    scheduler    = get_linear_schedule_with_warmup(optim, warmup_steps, total_steps)

    best_loss = float("inf")
    for epoch in range(cfg["epochs_pair"]):
        model.train()
        total_loss = 0.0
        for batch in loader:
            loss, _ = model(
                batch["input_ids"].to(device),
                batch["attention_mask"].to(device),
                batch["dist"].to(device),
                batch["order"].to(device),
                batch["has_but"].to(device),
                batch["label"].to(device),
            )
            optim.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optim.step()
            scheduler.step()
            total_loss += loss.item()

        avg = total_loss / len(loader)
        print(f"  [Pairing] epoch {epoch+1}/{cfg['epochs_pair']}  loss={avg:.4f}")
        if avg < best_loss:
            best_loss = avg
            torch.save(model.state_dict(), out_path)

    print(f"  Best pairing loss: {best_loss:.4f}  saved to {out_path}")
    return model


def train_va_regression(model, pair_examples, tokeniser, cfg, out_path="model_va_reg.pt"):
    """Train Stage 3A: VA regression (dropout disabled)."""
    device  = cfg["device"]
    model.to(device)
    dataset = VADataset(pair_examples, tokeniser, cfg["max_len"])
    loader  = DataLoader(dataset, batch_size=cfg["batch_size"], shuffle=True)
    optim   = make_optimizer(model, cfg["bio_lr"], cfg["head_lr"], cfg["weight_decay"])
    total_steps  = cfg["epochs_va"] * len(loader)
    warmup_steps = int(cfg["warmup_ratio"] * total_steps)
    scheduler    = get_linear_schedule_with_warmup(optim, warmup_steps, total_steps)

    best_loss = float("inf")
    for epoch in range(cfg["epochs_va"]):
        model.train()
        total_loss = 0.0
        for batch in loader:
            loss, _, _ = model(
                batch["input_ids"].to(device),
                batch["attention_mask"].to(device),
                batch["V"].to(device),
                batch["A"].to(device),
            )
            optim.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optim.step()
            scheduler.step()
            total_loss += loss.item()

        avg = total_loss / len(loader)
        print(f"  [VA-Reg] epoch {epoch+1}/{cfg['epochs_va']}  loss={avg:.4f}")
        if avg < best_loss:
            best_loss = avg
            torch.save(model.state_dict(), out_path)

    print(f"  Best VA-Reg loss: {best_loss:.4f}  saved to {out_path}")
    return model


def train_va_classification(model, pair_examples, tokeniser, cfg, out_path="model_va_cls.pt"):
    """Train Stage 3B: VA classification (32 bins)."""
    device  = cfg["device"]
    model.to(device)
    dataset = VADataset(pair_examples, tokeniser, cfg["max_len"])
    loader  = DataLoader(dataset, batch_size=cfg["batch_size"], shuffle=True)
    optim   = make_optimizer(model, cfg["bio_lr"], cfg["head_lr"], cfg["weight_decay"])
    total_steps  = cfg["epochs_va"] * len(loader)
    warmup_steps = int(cfg["warmup_ratio"] * total_steps)
    scheduler    = get_linear_schedule_with_warmup(optim, warmup_steps, total_steps)

    best_loss = float("inf")
    for epoch in range(cfg["epochs_va"]):
        model.train()
        total_loss = 0.0
        for batch in loader:
            loss, _, _ = model(
                batch["input_ids"].to(device),
                batch["attention_mask"].to(device),
                batch["V_bin"].to(device),
                batch["A_bin"].to(device),
            )
            optim.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optim.step()
            scheduler.step()
            total_loss += loss.item()

        avg = total_loss / len(loader)
        print(f"  [VA-Cls] epoch {epoch+1}/{cfg['epochs_va']}  loss={avg:.4f}")
        if avg < best_loss:
            best_loss = avg
            torch.save(model.state_dict(), out_path)

    print(f"  Best VA-Cls loss: {best_loss:.4f}  saved to {out_path}")
    return model

In [19]:
# ═══════════════════════════════════════════════════════════════════════════
# 5.  INFERENCE
# ═══════════════════════════════════════════════════════════════════════════

def span_to_original_text(original_text: str, aug_text: str, tok_start: int,
                           tok_end: int, offsets: list, tokeniser) -> str:
    """
    Recover original case-sensitive text using offset_mapping.
    Falls back to tokeniser decode if offsets are unavailable.
    """
    if tok_start == 0 and tok_end == 0:  # NULL placeholder
        return "NULL"
    try:
        char_start = offsets[tok_start][0]
        char_end   = offsets[tok_end][1]
        span_text  = aug_text[char_start:char_end].strip()
        if span_text:
            return span_text
    except (IndexError, TypeError):
        pass
    return "NULL"


def predict_sentence(rec, extract_model, pair_model, va_reg_model, va_cls_model,
                     tokeniser, cfg):
    """
    Run the full inference pipeline for one processed record.
    Returns list of {Aspect, Opinion, VA} dicts.
    """
    device = cfg["device"]

    input_ids      = rec["input_ids"].unsqueeze(0).to(device)
    attention_mask = rec["attention_mask"].unsqueeze(0).to(device)

    # Stage 1: extract spans
    extract_model.eval()
    asp_spans, opi_spans = extract_model.predict_spans(
        input_ids, attention_mask,
        threshold=cfg["span_threshold"],
        bio_thr=cfg["bio_proposal_thr"],
    )

    if not asp_spans and not opi_spans:
        # fallback: NULL+NULL pair
        asp_spans = [(0, 0)]
        opi_spans = [(0, 0)]

    # Stage 2: score all (asp, opi) pairs
    triplets = []
    pair_model.eval()
    va_reg_model.eval()
    va_cls_model.eval()

    for (asi, aej), (osi, oej) in product(asp_spans, opi_spans):
        toks = rec["tokens_decoded"]
        asp_text_tok = "".join(toks[asi: aej + 1]).strip()
        opi_text_tok = "".join(toks[osi: oej + 1]).strip()

        combined = rec["aug_text"] + " [SEP] " + asp_text_tok + " [SEP] " + opi_text_tok
        enc = tokeniser(
            combined,
            max_length=cfg["max_len"],
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        enc_ids  = enc["input_ids"].to(device)
        enc_mask = enc["attention_mask"].to(device)

        dist  = torch.tensor([[abs(asi - osi) / max(cfg["max_len"], 1)]], device=device)
        order = torch.tensor([[1.0 if asi < osi else 0.0]], device=device)
        hb    = torch.tensor([[has_but_between(toks, asi, osi)]], device=device)

        with torch.no_grad():
            _, pair_logits = pair_model(enc_ids, enc_mask, dist, order, hb)
            pair_prob = torch.softmax(pair_logits, dim=-1)[0, 1].item()

        if pair_prob < cfg["pair_threshold"]:
            continue

        # Stage 3: VA prediction
        with torch.no_grad():
            _, v_reg, a_reg = va_reg_model(enc_ids, enc_mask)
            _, v_logits, a_logits = va_cls_model(enc_ids, enc_mask)

        v_reg = v_reg.item()
        a_reg = a_reg.item()
        v_cls = bin_to_score(torch.argmax(v_logits, dim=-1).item())
        a_cls = bin_to_score(torch.argmax(a_logits, dim=-1).item())

        V_final = (v_reg + v_cls) / 2.0
        A_final = (a_reg + a_cls) / 2.0

        # clamp to valid range
        V_final = max(1.0, min(9.0, V_final))
        A_final = max(1.0, min(9.0, A_final))

        # recover original text
        asp_text = span_to_original_text(
            rec["raw_text"], rec["aug_text"], asi, aej, rec["offsets"], tokeniser)
        opi_text = span_to_original_text(
            rec["raw_text"], rec["aug_text"], osi, oej, rec["offsets"], tokeniser)

        triplets.append({
            "Aspect":  asp_text,
            "Opinion": opi_text,
            "VA":      f"{V_final:.2f}#{A_final:.2f}",
        })

    return triplets

In [20]:
# ═══════════════════════════════════════════════════════════════════════════
# 6.  EVALUATION
# ═══════════════════════════════════════════════════════════════════════════

def evaluate(records, extract_model, pair_model, va_reg_model, va_cls_model,
             tokeniser, cfg):
    """
    Compute VA-T-F1, V-T-F1, A-T-F1.
    A triplet is correct if:
      1. Aspect text matches exactly (case-sensitive)
      2. Opinion text matches exactly (case-sensitive)
      3. round(V_pred) == round(V_gold)
      4. round(A_pred) == round(A_gold)
    """
    total_pred = total_gold = total_correct_va = 0
    total_correct_v = total_correct_a = 0

    for rec in records:
        pred_triplets = predict_sentence(
            rec, extract_model, pair_model, va_reg_model, va_cls_model,
            tokeniser, cfg,
        )
        gold_triplets = [
            (gp[0], gp[1], gp[2], gp[3])
            for (asp_span, opi_span, V, A) in rec["gold_pairs"]
            for gp in [(
                span_to_original_text(rec["raw_text"], rec["aug_text"],
                                      asp_span[0], asp_span[1], rec["offsets"], tokeniser),
                span_to_original_text(rec["raw_text"], rec["aug_text"],
                                      opi_span[0], opi_span[1], rec["offsets"], tokeniser),
                V, A,
            )]
        ]

        total_pred += len(pred_triplets)
        total_gold += len(gold_triplets)

        for pt in pred_triplets:
            pV, pA = parse_va(pt["VA"])
            for (gAsp, gOpi, gV, gA) in gold_triplets:
                asp_match = pt["Aspect"]  == gAsp
                opi_match = pt["Opinion"] == gOpi
                v_match   = round(pV) == round(gV)
                a_match   = round(pA) == round(gA)
                if asp_match and opi_match and v_match and a_match:
                    total_correct_va += 1
                    total_correct_v  += 1
                    total_correct_a  += 1
                    break
                elif asp_match and opi_match and v_match:
                    total_correct_v += 1
                elif asp_match and opi_match and a_match:
                    total_correct_a += 1

    def f1(correct, pred, gold):
        if pred == 0 or gold == 0:
            return 0.0
        p = correct / pred
        r = correct / gold
        return 2 * p * r / (p + r) if (p + r) > 0 else 0.0

    va_f1 = f1(total_correct_va, total_pred, total_gold)
    v_f1  = f1(total_correct_v,  total_pred, total_gold)
    a_f1  = f1(total_correct_a,  total_pred, total_gold)

    print(f"\n  Evaluation Results:")
    print(f"    V-T-F1  = {v_f1:.4f}")
    print(f"    A-T-F1  = {a_f1:.4f}")
    print(f"    VA-T-F1 = {va_f1:.4f}  ← main metric")
    print(f"    (pred={total_pred}, gold={total_gold}, correct={total_correct_va})\n")
    return va_f1

In [21]:
# ═══════════════════════════════════════════════════════════════════════════
# 7.  MAIN
# ═══════════════════════════════════════════════════════════════════════════

def main():
    set_seed(CFG["seed"])
    device = CFG["device"]
    print(f"Device: {device}")
    print(f"Model:  {CFG['model_name']}")

    # ── Tokeniser ────────────────────────────────────────────────────────
    print("\n[1/7] Loading tokeniser...")
    tokeniser = AutoTokenizer.from_pretrained(CFG["model_name"])
    tokeniser.add_special_tokens(
        {"additional_special_tokens": ["[NULL]", "[LAP]", "[REST]"]}
    )
    print(f"  Vocab size after adding special tokens: {len(tokeniser)}")

    # ── Load & Process Data ───────────────────────────────────────────────
    print("\n[2/7] Loading and processing data...")
    lap_raw  = load_jsonl(LAP_FILE)
    rest_raw = load_jsonl(REST_FILE)
    print(f"  Laptop records:     {len(lap_raw)}")
    print(f"  Restaurant records: {len(rest_raw)}")

    lap_recs  = process_records(lap_raw,  "lap",  tokeniser, CFG["max_len"])
    rest_recs = process_records(rest_raw, "rest", tokeniser, CFG["max_len"])
    all_recs  = lap_recs + rest_recs
    random.shuffle(all_recs)

    # Simple 90/10 split for dev evaluation
    split = int(0.9 * len(all_recs))
    train_recs = all_recs[:split]
    dev_recs   = all_recs[split:]
    print(f"  Train: {len(train_recs)}  Dev: {len(dev_recs)}")

    # ── Build negative pairs ──────────────────────────────────────────────
    print("\n[3/7] Building negative pairs...")
    pair_examples = build_negative_pairs(train_recs, CFG["neg_ratio"])
    pos = sum(1 for p in pair_examples if p["label"] == 1)
    neg = sum(1 for p in pair_examples if p["label"] == 0)
    print(f"  Pair examples: {len(pair_examples)} (pos={pos}, neg={neg})")

    # ── Train Stage 1: Extraction ─────────────────────────────────────────
    print("\n[4/7] Training extraction model (BIO + span heads)...")
    extract_model = ExtractionModel(CFG)
    extract_model.encoder.resize_token_embeddings(len(tokeniser))
    train_extraction(extract_model, train_recs, CFG, "model_extract.pt")

    # ── Train Stage 2: Pairing ────────────────────────────────────────────
    print("\n[5/7] Training pairing classifier...")
    pair_model = PairingModel(CFG)
    pair_model.encoder.resize_token_embeddings(len(tokeniser))
    train_pairing(pair_model, pair_examples, tokeniser, CFG, "model_pair.pt")

    # ── Train Stage 3: VA Heads ───────────────────────────────────────────
    print("\n[6/7] Training VA heads...")
    va_reg_model = VARegressionModel(CFG)
    va_reg_model.encoder.resize_token_embeddings(len(tokeniser))
    train_va_regression(va_reg_model, pair_examples, tokeniser, CFG, "model_va_reg.pt")

    va_cls_model = VAClassificationModel(CFG)
    va_cls_model.encoder.resize_token_embeddings(len(tokeniser))
    train_va_classification(va_cls_model, pair_examples, tokeniser, CFG, "model_va_cls.pt")

    # ── Load best checkpoints ─────────────────────────────────────────────
    extract_model.load_state_dict(torch.load("model_extract.pt", map_location=device))
    pair_model.load_state_dict(torch.load("model_pair.pt", map_location=device))
    va_reg_model.load_state_dict(torch.load("model_va_reg.pt", map_location=device))
    va_cls_model.load_state_dict(torch.load("model_va_cls.pt", map_location=device))

    extract_model.to(device)
    pair_model.to(device)
    va_reg_model.to(device)
    va_cls_model.to(device)

    # ── Evaluate ──────────────────────────────────────────────────────────
    print("\n[7/7] Evaluating on dev set...")
    va_f1 = evaluate(
        dev_recs, extract_model, pair_model, va_reg_model, va_cls_model,
        tokeniser, CFG,
    )

    # ── Write sample predictions ──────────────────────────────────────────
    print("\nSample predictions (first 5 dev sentences):")
    for rec in dev_recs[:5]:
        triplets = predict_sentence(
            rec, extract_model, pair_model, va_reg_model, va_cls_model,
            tokeniser, CFG,
        )
        print(f"  ID: {rec['id']}")
        print(f"  Text: {rec['raw_text'][:80]}")
        print(f"  Predicted: {triplets}")
        print(f"  Gold pairs: {rec['gold_pairs']}")
        print()

    print(f"Final VA-T-F1: {va_f1:.4f}")


if __name__ == "__main__":
    main()

AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
"""
ASTE (Aspect Sentiment Triplet Extraction) with Valence-Arousal Regression
==========================================================================
Run this as the FIRST cell after kernel restart.
If you get CUDA errors, restart the kernel completely before re-running.

Install:  !pip install transformers==4.40.0 accelerate sentencepiece --quiet
"""

# ── restart guard: run this block first in a fresh kernel ──────────────────
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"   # synchronous CUDA errors for debugging
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import json
import math
import random
import warnings
from itertools import product
from pathlib import Path

warnings.filterwarnings("ignore")

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import (
    AutoModel,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
)

# ── Paths (Kaggle) ──────────────────────────────────────────────────────────
DATA_DIR  = Path("/kaggle/input/datasets/chaitanyajx1/datasetnlp")
REST_FILE = DATA_DIR / "restaurant_train.jsonl"
LAP_FILE  = DATA_DIR / "laptop_train.jsonl"

# ── Config ───────────────────────────────────────────────────────────────────
CFG = {
    "model_name":        "microsoft/deberta-v3-base",
    "max_len":           128,
    "hidden_size":       768,       # use 1024 for deberta-v3-large
    "width_emb_size":    64,
    "span_max_len":      8,
    "bio_classes":       5,         # O B-Asp I-Asp B-Opi I-Opi
    "num_bins":          32,        # VA bins (0.25 each)
    "bio_lr":            2e-5,
    "head_lr":           5e-4,
    "weight_decay":      0.01,
    "warmup_ratio":      0.1,
    "batch_size":        8,         # reduced to avoid OOM
    "epochs_extract":    10,
    "epochs_pair":       8,
    "epochs_va":         8,
    "pair_threshold":    0.5,
    "span_threshold":    0.5,
    "bio_proposal_thr":  0.3,
    "seed":              42,
    "neg_ratio":         3,
}

CFG["device"] = "cuda" if torch.cuda.is_available() else "cpu"

CONTRASTIVE = {"but", "however", "although", "though", "yet",
               "while", "whereas", "except", "despite", "still"}

BIO_O, BIO_B_ASP, BIO_I_ASP, BIO_B_OPI, BIO_I_OPI = 0, 1, 2, 3, 4

# ── Reproducibility ──────────────────────────────────────────────────────────
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        try:
            torch.cuda.manual_seed_all(seed)
        except Exception as e:
            print(f"  Warning: could not set CUDA seed: {e}")
            print("  Restart kernel if CUDA errors persist")


# ═══════════════════════════════════════════════════════════════════════════
# 1.  DATA LOADING & PRE-PROCESSING
# ═══════════════════════════════════════════════════════════════════════════

def load_jsonl(path: Path):
    records = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


def find_span_tokens(aug_text: str, span: str, offsets: list):
    """Find token (start, end) indices (inclusive) for span inside aug_text."""
    if not span or span == "NULL":
        return None
    idx = aug_text.find(span)
    if idx == -1:
        idx = aug_text.lower().find(span.lower())
    if idx == -1:
        return None
    char_start = idx
    char_end   = idx + len(span) - 1

    tok_start = tok_end = None
    for i, (cs, ce) in enumerate(offsets):
        if cs == 0 and ce == 0:
            continue
        if tok_start is None and cs <= char_start < ce:
            tok_start = i
        if cs <= char_end < ce:
            tok_end = i

    if tok_start is None or tok_end is None:
        return None
    return (tok_start, tok_end)


def build_bio_labels(seq_len: int, asp_spans, opi_spans):
    labels = [BIO_O] * seq_len
    for span in asp_spans:
        if span is None:
            continue
        s, e = span
        if s < seq_len:
            labels[s] = BIO_B_ASP
        for i in range(s + 1, min(e + 1, seq_len)):
            labels[i] = BIO_I_ASP
    for span in opi_spans:
        if span is None:
            continue
        s, e = span
        if s < seq_len:
            labels[s] = BIO_B_OPI
        for i in range(s + 1, min(e + 1, seq_len)):
            labels[i] = BIO_I_OPI
    return labels


def va_to_bin(score: float) -> int:
    return max(0, min(31, int((float(score) - 1.0) / 0.25)))


def bin_to_score(bin_idx: int) -> float:
    return 1.0 + bin_idx * 0.25 + 0.125


def parse_va(va_str: str):
    parts = str(va_str).split("#")
    try:
        return float(parts[0]), float(parts[1])
    except Exception:
        return 5.0, 5.0


def has_but_between(token_strings: list, start: int, end: int) -> float:
    lo, hi = min(start, end), max(start, end)
    for i in range(lo + 1, hi):
        if i < len(token_strings):
            tok = token_strings[i].strip().lower().strip("##")
            if tok in CONTRASTIVE:
                return 1.0
    return 0.0


def tokens_to_text(token_strings: list, s: int, e: int) -> str:
    """Join decoded tokens, strip subword ## prefixes."""
    pieces = []
    for i in range(s, min(e + 1, len(token_strings))):
        t = token_strings[i]
        if t.startswith("##"):
            t = t[2:]
        pieces.append(t)
    return "".join(pieces).strip()


def process_records(raw_records, domain: str, tokeniser, max_len: int):
    prefix = "[NULL] [LAP] " if domain == "lap" else "[NULL] [REST] "
    out = []

    for rec in raw_records:
        text  = rec.get("Text", "").strip()
        aug   = prefix + text
        quads = rec.get("Quadruplet", [])

        enc = tokeniser(
            aug,
            max_length=max_len,
            padding="max_length",
            truncation=True,
            return_offsets_mapping=True,
            return_tensors="pt",
        )
        input_ids      = enc["input_ids"].squeeze(0)
        attention_mask = enc["attention_mask"].squeeze(0)
        offsets        = enc["offset_mapping"].squeeze(0).tolist()

        token_strings = []
        for tid in input_ids.tolist():
            try:
                token_strings.append(tokeniser.decode([tid]))
            except Exception:
                token_strings.append("")

        asp_spans, opi_spans, gold_pairs = [], [], []

        for quad in quads:
            asp_text = quad.get("Aspect", "NULL") or "NULL"
            opi_text = quad.get("Opinion", "NULL") or "NULL"
            va_str   = quad.get("VA", "5.0#5.0")
            V, A     = parse_va(va_str)

            asp_span = (0, 0) if asp_text == "NULL" else find_span_tokens(aug, asp_text, offsets)
            if asp_span is None:
                asp_span = (0, 0)

            opi_span = (0, 0) if opi_text == "NULL" else find_span_tokens(aug, opi_text, offsets)
            if opi_span is None:
                opi_span = (0, 0)

            asp_spans.append(asp_span)
            opi_spans.append(opi_span)
            gold_pairs.append((asp_span, opi_span, V, A))

        bio_labels = build_bio_labels(max_len, asp_spans, opi_spans)

        out.append({
            "id":             rec.get("ID", ""),
            "raw_text":       text,
            "aug_text":       aug,
            "input_ids":      input_ids,
            "attention_mask": attention_mask,
            "bio_labels":     torch.tensor(bio_labels, dtype=torch.long),
            "gold_asp_spans": asp_spans,
            "gold_opi_spans": opi_spans,
            "gold_pairs":     gold_pairs,
            "offsets":        offsets,
            "token_strings":  token_strings,
            "domain":         domain,
        })

    return out


def build_pair_examples(all_records, neg_ratio: int = 3):
    positives = []
    negatives = []

    for rec in all_records:
        for (asp_span, opi_span, V, A) in rec["gold_pairs"]:
            positives.append({
                "aug_text":      rec["aug_text"],
                "input_ids":     rec["input_ids"],
                "attention_mask":rec["attention_mask"],
                "asp_span":      asp_span,
                "opi_span":      opi_span,
                "token_strings": rec["token_strings"],
                "label": 1, "V": V, "A": A,
            })

    for rec in all_records:
        gp_set = {(a, o) for (a, o, _, _) in rec["gold_pairs"]}
        for asp in rec["gold_asp_spans"]:
            for opi in rec["gold_opi_spans"]:
                if (asp, opi) not in gp_set:
                    negatives.append({
                        "aug_text":      rec["aug_text"],
                        "input_ids":     rec["input_ids"],
                        "attention_mask":rec["attention_mask"],
                        "asp_span":      asp,
                        "opi_span":      opi,
                        "token_strings": rec["token_strings"],
                        "label": 0, "V": 5.0, "A": 5.0,
                    })

    need = max(0, len(positives) * neg_ratio - len(negatives))
    for _ in range(need):
        r1 = random.choice(all_records)
        r2 = random.choice(all_records)
        if r1["gold_asp_spans"] and r2["gold_opi_spans"]:
            asp = random.choice(r1["gold_asp_spans"])
            opi = random.choice(r2["gold_opi_spans"])
            negatives.append({
                "aug_text":      r1["aug_text"],
                "input_ids":     r1["input_ids"],
                "attention_mask":r1["attention_mask"],
                "asp_span":      asp,
                "opi_span":      opi,
                "token_strings": r1["token_strings"],
                "label": 0, "V": 5.0, "A": 5.0,
            })

    all_ex = positives + negatives
    random.shuffle(all_ex)
    return all_ex


# ═══════════════════════════════════════════════════════════════════════════
# 2.  DATASETS
# ═══════════════════════════════════════════════════════════════════════════

class ExtractionDataset(Dataset):
    def __init__(self, records):
        self.records = records

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        r = self.records[idx]
        return {
            "input_ids":      r["input_ids"],
            "attention_mask": r["attention_mask"],
            "bio_labels":     r["bio_labels"],
            "gold_asp_spans": r["gold_asp_spans"],
            "gold_opi_spans": r["gold_opi_spans"],
        }


def extraction_collate(batch):
    return {
        "input_ids":      torch.stack([b["input_ids"]      for b in batch]),
        "attention_mask": torch.stack([b["attention_mask"] for b in batch]),
        "bio_labels":     torch.stack([b["bio_labels"]     for b in batch]),
        "gold_asp_spans": [b["gold_asp_spans"] for b in batch],
        "gold_opi_spans": [b["gold_opi_spans"] for b in batch],
    }


class PairVADataset(Dataset):
    def __init__(self, examples, tokeniser, max_len):
        self.examples  = examples
        self.tokeniser = tokeniser
        self.max_len   = max_len

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        ex  = self.examples[idx]
        asi, aej = ex["asp_span"]
        osi, oej = ex["opi_span"]
        toks = ex["token_strings"]

        asp_text = tokens_to_text(toks, asi, aej) if (asi, aej) != (0, 0) else "NULL"
        opi_text = tokens_to_text(toks, osi, oej) if (osi, oej) != (0, 0) else "NULL"

        combined = ex["aug_text"] + " [SEP] " + asp_text + " [SEP] " + opi_text
        enc = self.tokeniser(
            combined,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )

        dist  = abs(asi - osi) / max(self.max_len, 1)
        order = 1.0 if asi <= osi else 0.0
        hb    = has_but_between(toks, asi, osi)

        V, A = float(ex["V"]), float(ex["A"])
        return {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "dist":           torch.tensor([dist],  dtype=torch.float),
            "order":          torch.tensor([order], dtype=torch.float),
            "has_but":        torch.tensor([hb],    dtype=torch.float),
            "label":          torch.tensor(ex["label"], dtype=torch.long),
            "V":              torch.tensor(V, dtype=torch.float),
            "A":              torch.tensor(A, dtype=torch.float),
            "V_bin":          torch.tensor(va_to_bin(V), dtype=torch.long),
            "A_bin":          torch.tensor(va_to_bin(A), dtype=torch.long),
        }


# ═══════════════════════════════════════════════════════════════════════════
# 3.  MODELS
# ═══════════════════════════════════════════════════════════════════════════

class ExtractionModel(nn.Module):
    def __init__(self, cfg, vocab_size):
        super().__init__()
        self.encoder   = AutoModel.from_pretrained(cfg["model_name"])
        self.encoder.resize_token_embeddings(vocab_size)
        H = cfg["hidden_size"]
        W = cfg["width_emb_size"]
        self.width_emb = nn.Embedding(cfg["span_max_len"] + 1, W)
        self.bio_head  = nn.Linear(H, cfg["bio_classes"])
        self.asp_head  = nn.Sequential(
            nn.Linear(2*H+W, 256), nn.ReLU(), nn.Dropout(0.3), nn.Linear(256, 1))
        self.opi_head  = nn.Sequential(
            nn.Linear(2*H+W, 256), nn.ReLU(), nn.Dropout(0.3), nn.Linear(256, 1))

    def get_hidden(self, input_ids, attention_mask):
        return self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        ).last_hidden_state

    def span_vec(self, h_b, i: int, j: int):
        L = h_b.size(0)
        i = max(0, min(i, L-1))
        j = max(0, min(j, L-1))
        length = min(j - i, CFG["span_max_len"])
        w = self.width_emb(torch.tensor(length, device=h_b.device))
        return torch.cat([h_b[i], h_b[j], w], dim=-1)

    def forward(self, input_ids, attention_mask,
                bio_labels=None, gold_asp_spans=None, gold_opi_spans=None):
        h = self.get_hidden(input_ids, attention_mask)
        B = h.size(0)

        bio_logits = self.bio_head(h)
        loss_bio   = torch.tensor(0.0, device=h.device)
        if bio_labels is not None:
            loss_bio = F.cross_entropy(
                bio_logits.view(-1, CFG["bio_classes"]),
                bio_labels.view(-1),
                ignore_index=-100,
            )

        loss_asp = torch.tensor(0.0, device=h.device)
        loss_opi = torch.tensor(0.0, device=h.device)
        n_asp = n_opi = 0

        if gold_asp_spans is not None:
            for b in range(B):
                h_b = h[b]
                for (si, ei) in gold_asp_spans[b]:
                    sv    = self.span_vec(h_b, si, ei)
                    score = torch.sigmoid(self.asp_head(sv.unsqueeze(0))).squeeze()
                    loss_asp = loss_asp + F.binary_cross_entropy(
                        score, torch.ones(1, device=h.device).squeeze())
                    n_asp += 1
                for (si, ei) in gold_opi_spans[b]:
                    sv    = self.span_vec(h_b, si, ei)
                    score = torch.sigmoid(self.opi_head(sv.unsqueeze(0))).squeeze()
                    loss_opi = loss_opi + F.binary_cross_entropy(
                        score, torch.ones(1, device=h.device).squeeze())
                    n_opi += 1

        if n_asp > 0: loss_asp = loss_asp / n_asp
        if n_opi > 0: loss_opi = loss_opi / n_opi

        return loss_bio + loss_asp + loss_opi, bio_logits

    @torch.no_grad()
    def predict_spans(self, input_ids, attention_mask):
        h_b       = self.get_hidden(input_ids, attention_mask)[0]
        bio_probs = torch.softmax(self.bio_head(h_b), dim=-1)
        L         = h_b.size(0)
        thr       = CFG["bio_proposal_thr"]
        span_thr  = CFG["span_threshold"]

        cands_asp, cands_opi = set(), set()
        for i in range(L):
            if bio_probs[i, BIO_B_ASP].item() > thr:
                for j in range(i, min(i + CFG["span_max_len"], L)):
                    cands_asp.add((i, j))
            if bio_probs[i, BIO_B_OPI].item() > thr:
                for j in range(i, min(i + CFG["span_max_len"], L)):
                    cands_opi.add((i, j))

        final_asp, final_opi = [], []
        for (si, ei) in cands_asp:
            sv = self.span_vec(h_b, si, ei)
            if torch.sigmoid(self.asp_head(sv.unsqueeze(0))).item() >= span_thr:
                final_asp.append((si, ei))
        for (si, ei) in cands_opi:
            sv = self.span_vec(h_b, si, ei)
            if torch.sigmoid(self.opi_head(sv.unsqueeze(0))).item() >= span_thr:
                final_opi.append((si, ei))

        return final_asp, final_opi


class PairingModel(nn.Module):
    def __init__(self, cfg, vocab_size):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(cfg["model_name"])
        self.encoder.resize_token_embeddings(vocab_size)
        self.clf = nn.Linear(cfg["hidden_size"] + 3, 2)

    def forward(self, input_ids, attention_mask, dist, order, has_but, labels=None):
        cls    = self.encoder(input_ids=input_ids,
                              attention_mask=attention_mask).last_hidden_state[:, 0, :]
        feats  = torch.cat([cls, dist, order, has_but], dim=-1)
        logits = self.clf(feats)
        loss   = F.cross_entropy(logits, labels) if labels is not None else None
        return loss, logits


class VARegressionModel(nn.Module):
    def __init__(self, cfg, vocab_size):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(cfg["model_name"])
        self.encoder.resize_token_embeddings(vocab_size)
        H = cfg["hidden_size"]
        self.v_head = nn.Linear(H, 1)
        self.a_head = nn.Linear(H, 1)
        # disable dropout for numerical stability
        for m in self.encoder.modules():
            if isinstance(m, nn.Dropout):
                m.p = 0.0

    def forward(self, input_ids, attention_mask, V=None, A=None):
        cls   = self.encoder(input_ids=input_ids,
                             attention_mask=attention_mask).last_hidden_state[:, 0, :]
        v_hat = self.v_head(cls).squeeze(-1)
        a_hat = self.a_head(cls).squeeze(-1)
        loss  = (F.mse_loss(v_hat, V) + F.mse_loss(a_hat, A)) if V is not None else None
        return loss, v_hat, a_hat


class VAClassificationModel(nn.Module):
    def __init__(self, cfg, vocab_size):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(cfg["model_name"])
        self.encoder.resize_token_embeddings(vocab_size)
        H = cfg["hidden_size"]
        self.v_head = nn.Linear(H, cfg["num_bins"])
        self.a_head = nn.Linear(H, cfg["num_bins"])

    def forward(self, input_ids, attention_mask, V_bin=None, A_bin=None):
        cls      = self.encoder(input_ids=input_ids,
                                attention_mask=attention_mask).last_hidden_state[:, 0, :]
        v_logits = self.v_head(cls)
        a_logits = self.a_head(cls)
        loss     = (F.cross_entropy(v_logits, V_bin) +
                    F.cross_entropy(a_logits, A_bin)) if V_bin is not None else None
        return loss, v_logits, a_logits


# ═══════════════════════════════════════════════════════════════════════════
# 4.  TRAINING
# ═══════════════════════════════════════════════════════════════════════════

def make_optimizer(model, enc_lr, head_lr, wd):
    enc_ids = {id(p) for p in model.encoder.parameters()}
    heads   = [p for p in model.parameters() if id(p) not in enc_ids]
    return torch.optim.AdamW(
        [{"params": list(model.encoder.parameters()), "lr": enc_lr},
         {"params": heads, "lr": head_lr}],
        weight_decay=wd,
    )


def run_train_loop(model, loader, n_epochs, cfg, label, out_path, forward_fn):
    device       = cfg["device"]
    model.to(device)
    optim        = make_optimizer(model, cfg["bio_lr"], cfg["head_lr"], cfg["weight_decay"])
    total_steps  = n_epochs * len(loader)
    warmup_steps = max(1, int(cfg["warmup_ratio"] * total_steps))
    sched        = get_linear_schedule_with_warmup(optim, warmup_steps, total_steps)
    best_loss    = float("inf")

    for epoch in range(n_epochs):
        model.train()
        running = 0.0
        for batch in loader:
            try:
                loss = forward_fn(model, batch, device)
                if loss is None or torch.isnan(loss):
                    continue
                optim.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optim.step()
                sched.step()
                running += loss.item()
            except RuntimeError as e:
                print(f"  [WARNING] skipped batch: {e}")
                optim.zero_grad()
                continue

        avg = running / max(len(loader), 1)
        print(f"  [{label}] epoch {epoch+1}/{n_epochs}  loss={avg:.4f}")
        if avg < best_loss:
            best_loss = avg
            torch.save(model.state_dict(), out_path)

    print(f"  Saved best {label} -> {out_path}  (loss={best_loss:.4f})")
    return model


def fwd_extract(model, batch, device):
    loss, _ = model(
        batch["input_ids"].to(device),
        batch["attention_mask"].to(device),
        batch["bio_labels"].to(device),
        batch["gold_asp_spans"],
        batch["gold_opi_spans"],
    )
    return loss

def fwd_pair(model, batch, device):
    loss, _ = model(
        batch["input_ids"].to(device),
        batch["attention_mask"].to(device),
        batch["dist"].to(device),
        batch["order"].to(device),
        batch["has_but"].to(device),
        batch["label"].to(device),
    )
    return loss

def fwd_va_reg(model, batch, device):
    loss, _, _ = model(
        batch["input_ids"].to(device),
        batch["attention_mask"].to(device),
        batch["V"].to(device),
        batch["A"].to(device),
    )
    return loss

def fwd_va_cls(model, batch, device):
    loss, _, _ = model(
        batch["input_ids"].to(device),
        batch["attention_mask"].to(device),
        batch["V_bin"].to(device),
        batch["A_bin"].to(device),
    )
    return loss


# ═══════════════════════════════════════════════════════════════════════════
# 5.  INFERENCE
# ═══════════════════════════════════════════════════════════════════════════

def recover_span_text(aug_text: str, tok_start: int, tok_end: int, offsets: list) -> str:
    if tok_start == 0 and tok_end == 0:
        return "NULL"
    try:
        cs = offsets[tok_start][0]
        ce = offsets[tok_end][1]
        t  = aug_text[cs:ce].strip()
        if t:
            return t
    except (IndexError, TypeError):
        pass
    return "NULL"


@torch.no_grad()
def predict_one(rec, extract_m, pair_m, va_reg_m, va_cls_m, tokeniser, cfg):
    device = cfg["device"]
    ids    = rec["input_ids"].unsqueeze(0).to(device)
    mask   = rec["attention_mask"].unsqueeze(0).to(device)

    extract_m.eval()
    asp_spans, opi_spans = extract_m.predict_spans(ids, mask)
    if not asp_spans:
        asp_spans = [(0, 0)]
    if not opi_spans:
        opi_spans = [(0, 0)]

    triplets = []
    pair_m.eval(); va_reg_m.eval(); va_cls_m.eval()

    for (asi, aej), (osi, oej) in product(asp_spans, opi_spans):
        toks     = rec["token_strings"]
        asp_text = tokens_to_text(toks, asi, aej) if (asi,aej) != (0,0) else "NULL"
        opi_text = tokens_to_text(toks, osi, oej) if (osi,oej) != (0,0) else "NULL"

        combined = rec["aug_text"] + " [SEP] " + asp_text + " [SEP] " + opi_text
        enc = tokeniser(combined, max_length=cfg["max_len"], padding="max_length",
                        truncation=True, return_tensors="pt")
        e_ids  = enc["input_ids"].to(device)
        e_mask = enc["attention_mask"].to(device)

        dist  = torch.tensor([[abs(asi-osi)/max(cfg["max_len"],1)]], device=device)
        order = torch.tensor([[1.0 if asi<=osi else 0.0]], device=device)
        hb    = torch.tensor([[has_but_between(toks, asi, osi)]], device=device)

        _, pair_logits = pair_m(e_ids, e_mask, dist, order, hb)
        if torch.softmax(pair_logits, dim=-1)[0, 1].item() < cfg["pair_threshold"]:
            continue

        _, v_reg, a_reg = va_reg_m(e_ids, e_mask)
        _, v_log, a_log = va_cls_m(e_ids, e_mask)

        v_cls = bin_to_score(int(torch.argmax(v_log, dim=-1).item()))
        a_cls = bin_to_score(int(torch.argmax(a_log, dim=-1).item()))
        V_f   = max(1.0, min(9.0, (v_reg.item() + v_cls) / 2.0))
        A_f   = max(1.0, min(9.0, (a_reg.item() + a_cls) / 2.0))

        asp_orig = recover_span_text(rec["aug_text"], asi, aej, rec["offsets"])
        opi_orig = recover_span_text(rec["aug_text"], osi, oej, rec["offsets"])

        triplets.append({"Aspect": asp_orig, "Opinion": opi_orig,
                         "VA": f"{V_f:.2f}#{A_f:.2f}"})
    return triplets


# ═══════════════════════════════════════════════════════════════════════════
# 6.  EVALUATION
# ═══════════════════════════════════════════════════════════════════════════

def evaluate(records, extract_m, pair_m, va_reg_m, va_cls_m, tokeniser, cfg):
    total_pred = total_gold = 0
    correct_va = correct_v = correct_a = 0

    for rec in records:
        pred = predict_one(rec, extract_m, pair_m, va_reg_m, va_cls_m, tokeniser, cfg)
        gold = [(recover_span_text(rec["aug_text"], a[0], a[1], rec["offsets"]),
                 recover_span_text(rec["aug_text"], o[0], o[1], rec["offsets"]),
                 gV, gA)
                for (a, o, gV, gA) in rec["gold_pairs"]]

        total_pred += len(pred)
        total_gold += len(gold)

        for pt in pred:
            pV, pA = parse_va(pt["VA"])
            for (ga, go, gV, gA) in gold:
                a_ok  = pt["Aspect"]  == ga
                o_ok  = pt["Opinion"] == go
                v_ok  = round(pV) == round(gV)
                a2_ok = round(pA) == round(gA)
                if a_ok and o_ok and v_ok and a2_ok:
                    correct_va += 1
                    correct_v  += 1
                    correct_a  += 1
                    break
                if a_ok and o_ok and v_ok:  correct_v += 1
                if a_ok and o_ok and a2_ok: correct_a += 1

    def f1(c, p, g):
        if p == 0 or g == 0: return 0.0
        pr = c/p; re = c/g
        return 2*pr*re/(pr+re) if (pr+re) > 0 else 0.0

    vaf1 = f1(correct_va, total_pred, total_gold)
    vf1  = f1(correct_v,  total_pred, total_gold)
    af1  = f1(correct_a,  total_pred, total_gold)

    print(f"\n  Evaluation")
    print(f"    V-T-F1  = {vf1:.4f}")
    print(f"    A-T-F1  = {af1:.4f}")
    print(f"    VA-T-F1 = {vaf1:.4f}  <- main metric")
    print(f"    pred={total_pred}  gold={total_gold}  correct={correct_va}\n")
    return vaf1


# ═══════════════════════════════════════════════════════════════════════════
# 7.  MAIN
# ═══════════════════════════════════════════════════════════════════════════

def main():
    set_seed(CFG["seed"])
    device = CFG["device"]
    print(f"Device : {device}")
    print(f"Model  : {CFG['model_name']}")

    print("\n[1/7] Loading tokeniser...")
    tokeniser = AutoTokenizer.from_pretrained(CFG["model_name"])
    tokeniser.add_special_tokens(
        {"additional_special_tokens": ["[NULL]", "[LAP]", "[REST]"]}
    )
    vocab_size = len(tokeniser)
    print(f"  Vocab size: {vocab_size}")

    print("\n[2/7] Loading data...")
    lap_raw  = load_jsonl(LAP_FILE)
    rest_raw = load_jsonl(REST_FILE)
    print(f"  Laptop: {len(lap_raw)}  Restaurant: {len(rest_raw)}")

    print("  Processing laptop records...")
    lap_recs  = process_records(lap_raw,  "lap",  tokeniser, CFG["max_len"])
    print("  Processing restaurant records...")
    rest_recs = process_records(rest_raw, "rest", tokeniser, CFG["max_len"])
    all_recs  = lap_recs + rest_recs
    random.shuffle(all_recs)

    split      = int(0.9 * len(all_recs))
    train_recs = all_recs[:split]
    dev_recs   = all_recs[split:]
    print(f"  Train: {len(train_recs)}  Dev: {len(dev_recs)}")

    print("\n[3/7] Building pair examples...")
    pair_ex = build_pair_examples(train_recs, CFG["neg_ratio"])
    pos_n   = sum(1 for e in pair_ex if e["label"] == 1)
    neg_n   = sum(1 for e in pair_ex if e["label"] == 0)
    va_ex   = [e for e in pair_ex if e["label"] == 1]
    print(f"  Total: {len(pair_ex)}  (pos={pos_n}  neg={neg_n})")

    ext_loader  = DataLoader(ExtractionDataset(train_recs),
                             batch_size=CFG["batch_size"], shuffle=True,
                             collate_fn=extraction_collate)
    pair_loader = DataLoader(PairVADataset(pair_ex, tokeniser, CFG["max_len"]),
                             batch_size=CFG["batch_size"], shuffle=True)
    va_loader   = DataLoader(PairVADataset(va_ex, tokeniser, CFG["max_len"]),
                             batch_size=CFG["batch_size"], shuffle=True)

    print("\n[4/7] Training extraction model...")
    extract_m = ExtractionModel(CFG, vocab_size)
    run_train_loop(extract_m, ext_loader, CFG["epochs_extract"], CFG,
                   "Extract", "model_extract.pt", fwd_extract)

    print("\n[5/7] Training pairing classifier...")
    pair_m = PairingModel(CFG, vocab_size)
    run_train_loop(pair_m, pair_loader, CFG["epochs_pair"], CFG,
                   "Pairing", "model_pair.pt", fwd_pair)

    print("\n[6a/7] Training VA regression (dropout disabled)...")
    va_reg_m = VARegressionModel(CFG, vocab_size)
    run_train_loop(va_reg_m, va_loader, CFG["epochs_va"], CFG,
                   "VA-Reg", "model_va_reg.pt", fwd_va_reg)

    print("\n[6b/7] Training VA classification (32 bins)...")
    va_cls_m = VAClassificationModel(CFG, vocab_size)
    run_train_loop(va_cls_m, va_loader, CFG["epochs_va"], CFG,
                   "VA-Cls", "model_va_cls.pt", fwd_va_cls)

    print("\n[7/7] Loading best checkpoints and evaluating...")
    extract_m.load_state_dict(torch.load("model_extract.pt", map_location=device))
    pair_m.load_state_dict(torch.load("model_pair.pt",       map_location=device))
    va_reg_m.load_state_dict(torch.load("model_va_reg.pt",   map_location=device))
    va_cls_m.load_state_dict(torch.load("model_va_cls.pt",   map_location=device))

    for m in [extract_m, pair_m, va_reg_m, va_cls_m]:
        m.to(device)

    va_f1 = evaluate(dev_recs, extract_m, pair_m, va_reg_m, va_cls_m, tokeniser, CFG)

    print("Sample predictions:")
    for rec in dev_recs[:3]:
        preds = predict_one(rec, extract_m, pair_m, va_reg_m, va_cls_m, tokeniser, CFG)
        golds = [(recover_span_text(rec["aug_text"], a[0], a[1], rec["offsets"]),
                  recover_span_text(rec["aug_text"], o[0], o[1], rec["offsets"]),
                  f"{gV:.2f}#{gA:.2f}")
                 for (a, o, gV, gA) in rec["gold_pairs"]]
        print(f"  Text : {rec['raw_text'][:70]}")
        print(f"  Pred : {preds}")
        print(f"  Gold : {golds}")
        print()

    print(f"Final VA-T-F1 = {va_f1:.4f}")
    return va_f1


if __name__ == "__main__":
    main()

Device : cuda
Model  : microsoft/deberta-v3-base

[1/7] Loading tokeniser...
  Vocab size: 128004

[2/7] Loading data...
  Laptop: 4076  Restaurant: 2284
  Processing laptop records...
  Processing restaurant records...
  Train: 5724  Dev: 636

[3/7] Building pair examples...
  Total: 34080  (pos=8520  neg=25560)

[4/7] Training extraction model...


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  [WARNING] skipped batch: mat1 and mat2 must have the same dtype, but got Half and Float
  [WARNING] skipped batch: mat1 and mat2 must have the same dtype, but got Half and Float
  [WARNING] skipped batch: mat1 and mat2 must have the same dtype, but got Half and Float
  [WARNING] skipped batch: mat1 and mat2 must have the same dtype, but got Half and Float
  [WARNING] skipped batch: mat1 and mat2 must have the same dtype, but got Half and Float
  [WARNING] skipped batch: mat1 and mat2 must have the same dtype, but got Half and Float
  [WARNING] skipped batch: mat1 and mat2 must have the same dtype, but got Half and Float
  [WARNING] skipped batch: mat1 and mat2 must have the same dtype, but got Half and Float
  [WARNING] skipped batch: mat1 and mat2 must have the same dtype, but got Half and Float
  [WARNING] skipped batch: mat1 and mat2 must have the same dtype, but got Half and Float
  [WARNING] skipped batch: mat1 and mat2 must have the same dtype, but got Half and Float
  [WARNING

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
